### **Ejercicio N° 10**

El dataset `ventas.xlsx` contiene los registros de una serie de ventas realizadas en el último tiempo en un local de productos electrónicos. Por otra parte, cuenta con el dataset `clientes_base.xlsx`, el cual contiene información sobre los clientes registrados en dicho establecimiento. 

1. ¿Cuál fue el monto total de venta de productos iPad y MacBook?

2. Realice la unión de ambos DataFrames utilizando la operación que considere más adecuada y la columna `nombre_cliente` como *key.* ¿Qué observa en el DataFrame resultante?

3. Considerando que en `clientes_base.xlsx` los nombres de los clientes se encuentran exentos de errores ortográficos y tipográficos, ¿en qué porcentaje de los registros que conforman el dataset `ventas.xlsx` el nombre del cliente coincide con el de un cliente registrado?

4. Teniendo en cuenta lo observado en los ítems anteriores, utilice herramientas de *fuzzy joins* para realizar la unión de ambos datasets. ¿De qué ciudad es el cliente que más compras realizó en el local?

In [67]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [68]:
ventas = pd.read_excel('./datasets/ventas.xlsx')
ventas.set_index('id_venta',inplace=True)
ventas.head()

,nombre_cliente,producto,cantidad,precio_usd_producto
id_venta,,,,
C1,Juana Perez,Apple Watch Series 8,2,399
C2,Roberto Gomezz,Nintendo Switch,1,299
C3,Carla Gonzáles Cuispe,Bose QuietComfort 45,1,329
C4,Jorge Martinez,Acer Predator Helios 300,1,1599
C5,Mariano Rodriguéz,iPad Pro,1,1099


In [69]:
ventas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42 entries, C1 to C42
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   nombre_cliente       42 non-null     object
 1   producto             42 non-null     object
 2   cantidad             42 non-null     int64 
 3   precio_usd_producto  42 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 1.6+ KB


In [70]:
clientes=pd.read_excel('./datasets/clientes_base.xlsx')
clientes.set_index('id_cliente',inplace=True)
clientes.head()

,nombre_cliente,ciudad,email
id_cliente,,,
1,Lucia Fernandez,Villa María,luciaf2@mail.com
2,Carlos Gómez,Mendoza,carlosgomez@mail.com
3,Andrés Pérez,Corrientes,andresp3@mail.com
4,Roberto Gómez,Rosario,rgomez@mail.com
5,Roberto Gómez Acuña,Corrientes,robgoac@mail.com


In [71]:
clientes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 1 to 36
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   nombre_cliente  36 non-null     object
 1   ciudad          36 non-null     object
 2   email           36 non-null     object
dtypes: object(3)
memory usage: 1.1+ KB


### ¿Cuál fue el monto total de venta de productos iPad y MacBook?

In [72]:
ventas['monto']=ventas['cantidad']*ventas['precio_usd_producto']
ventas.head()

,nombre_cliente,producto,cantidad,precio_usd_producto,monto
id_venta,,,,,
C1,Juana Perez,Apple Watch Series 8,2,399,798
C2,Roberto Gomezz,Nintendo Switch,1,299,299
C3,Carla Gonzáles Cuispe,Bose QuietComfort 45,1,329,329
C4,Jorge Martinez,Acer Predator Helios 300,1,1599,1599
C5,Mariano Rodriguéz,iPad Pro,1,1099,1099


In [73]:
#filtro con startswith todos los iPad y Macbook
#ipad_y_macbook=ventas[ventas['producto'].str.startswith(('iPad','MacBook'), na=False)]
ipad_y_macbook=ventas[ventas['producto'].str.contains('iPad|MacBook')]
ipad_y_macbook

,nombre_cliente,producto,cantidad,precio_usd_producto,monto
id_venta,,,,,
C5,Mariano Rodriguéz,iPad Pro,1,1099,1099
C7,Maria Garcìa,MacBook Air,1,1249,1249
C16,Laura Martínez,iPad mini,1,559,559
C19,Andres Pérrez,MacBook Pro,1,1999,1999
C23,Andres Péres,iPad Air,2,599,1198
C39,Juan Perez,iPad mini,1,559,559


In [74]:
monto_ventas=ipad_y_macbook.groupby('producto')['monto'].sum()
monto=ipad_y_macbook['monto'].sum()
print(f'Total monto de ventas MacBooks y iPads: \n {monto_ventas} \nMonto total de ventas: ${monto}')

Total monto de ventas MacBooks y iPads: 
 producto
MacBook Air    1249
MacBook Pro    1999
iPad Air       1198
iPad Pro       1099
iPad mini      1118
Name: monto, dtype: int64 
Monto total de ventas: $6663


### Realice la unión de ambos DataFrames utilizando la operación que considere más adecuada y la columna nombre_cliente como key. ¿Qué observa en el DataFrame resultante?

In [75]:
union=pd.merge(ventas,clientes, on='nombre_cliente')
union.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   nombre_cliente       14 non-null     object
 1   producto             14 non-null     object
 2   cantidad             14 non-null     int64 
 3   precio_usd_producto  14 non-null     int64 
 4   monto                14 non-null     int64 
 5   ciudad               14 non-null     object
 6   email                14 non-null     object
dtypes: int64(3), object(4)
memory usage: 916.0+ bytes


In [76]:
union
#en la tabla ventas hay nombres mal escritos que no coinciden con los de la tabla clientes entonces no se muestran en la union 

,nombre_cliente,producto,cantidad,precio_usd_producto,monto,ciudad,email
0,Roberto Gómez Acuña,HP Spectre x360,1,1399,1399,Corrientes,robgoac@mail.com
1,María García,Amazon Echo Dot,3,49,147,Córdoba,mariagarcia@mail.com
2,Andrés Pérez Gollán,Google Pixel 7,2,599,1198,Mar del Plata,andresp@mail.com
3,Miguel Angel,Microsoft Surface Pro,1,899,899,Paraná,miguelangel2@mail.com
4,Roberto Gómez,Sony PlayStation 5,1,499,499,Rosario,rgomez@mail.com
5,Laura Martínez,iPad mini,1,559,559,San Juan,lauram@mail.com
6,Ana Fernández,GoPro Hero 11,1,399,399,Posadas,anafernandez@mail.com
7,Andres López Corti,Canon EOS R5,1,3899,3899,Rosario,andreslopez@mail.com
8,Lauriana Martínez,Razer Blade 15,1,1599,1599,Santa Fe,marlau@mail.com
9,Mariana Perez,Bose SoundLink,1,199,199,La Rioja,marianap@mail.com


### Considerando que en clientes_base.xlsx los nombres de los clientes se encuentran exentos de errores ortográficos y tipográficos, ¿en qué porcentaje de los registros que conforman el dataset ventas.xlsx el nombre del cliente coincide con el de un cliente registrado?


In [77]:
porcentaje = float(round((union['nombre_cliente'].count() / clientes['nombre_cliente'].count()) * 100, 2))

print(f'El porcentaje es: {porcentaje} %')

El porcentaje es: 38.89 %


### Teniendo en cuenta lo observado en los ítems anteriores, utilice herramientas de fuzzy joins para realizar la unión de ambos datasets. ¿De qué ciudad es el cliente que más compras realizó en el local?

In [78]:
nombres_oficiales = clientes['nombre_cliente'].tolist()

In [79]:
#Iteramos sobre los nombres mal escritos de la tabla de ventas
for index, row in ventas.iterrows():
    nombre_venta = row['nombre_cliente']
    #Encontramos la mejor coincidencia en df1 utilizando process.extractOne()
    best_match = process.extractOne(
        query=nombre_venta, 
        choices=clientes['nombre_cliente'], 
        scorer=fuzz.token_sort_ratio, 
        score_cutoff=70)
    
    # best_match devuelve una tupla: (Nombre_Coincidente, Puntaje_Similitud, Indice)
    if best_match:
        # Reemplazamos el nombre con errores por el nombre oficial (el elemento 0 de la tupla)
        ventas.at[index, 'nombre_cliente'] = best_match[0]

In [80]:
union_limpia = pd.merge(ventas, clientes, on='nombre_cliente', how='inner')
union_limpia.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   nombre_cliente       42 non-null     object
 1   producto             42 non-null     object
 2   cantidad             42 non-null     int64 
 3   precio_usd_producto  42 non-null     int64 
 4   monto                42 non-null     int64 
 5   ciudad               42 non-null     object
 6   email                42 non-null     object
dtypes: int64(3), object(4)
memory usage: 2.4+ KB


In [88]:
cliente_top=union_limpia.groupby('nombre_cliente')['monto'].sum().idxmax()
filtro=union_limpia[union_limpia['nombre_cliente']==cliente_top]
ciudad = filtro['ciudad'].iloc[0]
print(f'El cliente con mas compras es {cliente_top} de la ciudad de {ciudad}')

El cliente con mas compras es Andres López Corti de la ciudad de Rosario
